In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, recall_score
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import os
from tqdm import tqdm
import re
import random
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight

In [5]:
data_dir = '/content/TableClassifierQuarterlyWithNotes'
print(f"Data directory set to: {data_dir}")

Data directory set to: /content/TableClassifierQuarterlyWithNotes


In [1]:
def improved_html_preprocessing(html_content):
    """
    IMPROVED: Better HTML preprocessing that preserves financial document structure
    """
    soup = BeautifulSoup(html_content, 'html.parser')

    # Remove script and style elements
    for script in soup(["script", "style"]):
        script.decompose()

    # Extract text but preserve some structure - CRITICAL for classification
    text_parts = []

    # Look for tables (common in financial docs) - ADDRESSES Notes vs Others confusion
    tables = soup.find_all('table')
    for table in tables:
        table_text = table.get_text(separator=' | ', strip=True)
        if table_text and len(table_text) > 20:  # Filter out small tables
            text_parts.append(f"[TABLE] {table_text}")

    # Look for headers that indicate document type - CRITICAL classification signal
    for header_tag in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
        headers = soup.find_all(header_tag)
        for header in headers:
            header_text = header.get_text(strip=True)
            if header_text and len(header_text) > 3:
                text_parts.append(f"[HEADER] {header_text}")

    # Look for financial keywords in class names - HELPS Cash Flow detection
    financial_elements = soup.find_all(attrs={"class": re.compile(r"cash|flow|statement|income|balance", re.I)})
    for elem in financial_elements[:3]:  # Limit to avoid noise
        elem_text = elem.get_text(strip=True)
        if elem_text and len(elem_text) > 10:
            text_parts.append(f"[FINANCIAL_ELEMENT] {elem_text[:100]}")

    # Get remaining text
    remaining_text = soup.get_text(separator=' ', strip=True)

    # Financial document specific cleaning and normalization
    remaining_text = re.sub(r'\s+', ' ', remaining_text)  # Multiple spaces to single

    # Normalize patterns while keeping classification signals
    remaining_text = re.sub(r'\b\d{1,2}/\d{1,2}/\d{2,4}\b', '[DATE]', remaining_text)
    remaining_text = re.sub(r'\$[\d,]+\.?\d*', '[CURRENCY]', remaining_text)
    remaining_text = re.sub(r'\b\d{1,3}(?:,\d{3})+\b', '[NUMBER]', remaining_text)

    # Combine all parts - structure signals first, then content
    full_text = ' '.join(text_parts) + ' ' + remaining_text

    # Add document type indicators - CRITICAL for minority class detection
    text_lower = full_text.lower()

    # Cash flow specific patterns - ADDRESSES Cash Flow low recall
    if any(phrase in text_lower for phrase in ['cash flow', 'operating activities', 'investing activities', 'financing activities', 'cash provided', 'cash used']):
        full_text = '[CASH_FLOW_INDICATOR] ' + full_text

    # Income statement patterns
    elif any(phrase in text_lower for phrase in ['revenue', 'total revenues', 'net income', 'earnings per share', 'profit', 'income statement']):
        full_text = '[INCOME_INDICATOR] ' + full_text

    # Balance sheet patterns
    elif any(phrase in text_lower for phrase in ['total assets', 'total liabilities', 'shareholders equity', 'balance sheet']):
        full_text = '[BALANCE_INDICATOR] ' + full_text

    # Notes patterns - ADDRESSES Notes vs Others confusion
    elif any(phrase in text_lower for phrase in ['note ', 'notes to', 'footnote', 'see note', 'supplementary', 'disclosure']):
        full_text = '[NOTES_INDICATOR] ' + full_text

    return full_text.strip()

def load_financial_documents(data_dir):
    """
    Load HTML financial documents from directory structure
    """
    documents = []
    labels = []
    label_map = {
        'Income Statement': 0,
        'Balance Sheets': 1,
        'Cash Flow': 2,
        'Notes': 3,
        'Others': 4
    }


    for label_name, label_id in label_map.items():
        folder_path = os.path.join(data_dir, label_name)
        if os.path.exists(folder_path):
            files = [f for f in os.listdir(folder_path) if f.endswith('.html')]
            print(f"Processing {len(files)} files for {label_name}...")

            for filename in files:
                file_path = os.path.join(folder_path, filename)
                try:
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        html_content = f.read()


                    text = improved_html_preprocessing(html_content)

                    if text.strip():
                        documents.append(text)
                        labels.append(label_id)

                except Exception as e:
                    print(f"Error processing {filename}: {e}")
                    continue

    print(f"\nTotal documents loaded: {len(documents)}")
    return documents, labels, label_map

print("Preprocessing functions defined")

Preprocessing functions defined


In [6]:
documents, labels, label_map = load_financial_documents(data_dir)

# Show summary
print(f"\n DATASET SUMMARY:")
print(f"Total documents: {len(documents)}")

label_dist = Counter(labels)
print("\nOriginal Class Distribution:")
for label_name, label_id in label_map.items():
    print(f"  {label_name}: {label_dist[label_id]}")

print(f"\nSAMPLE DOCUMENTS:")
for i, doc in enumerate(documents[:3]):
    label_name = list(label_map.keys())[list(label_map.values()).index(labels[i])]
    print(f"Document {i+1} ({label_name}): {doc[:150]}...")

print("\nData loaded successfully")

Processing 317 files for Income Statement...
Processing 282 files for Balance Sheets...
Processing 36 files for Cash Flow...
Processing 702 files for Notes...
Processing 1236 files for Others...

Total documents loaded: 2573

 DATASET SUMMARY:
Total documents: 2573

Original Class Distribution:
  Income Statement: 317
  Balance Sheets: 282
  Cash Flow: 36
  Notes: 702
  Others: 1236

SAMPLE DOCUMENTS:
Document 1 (Income Statement): [INCOME_INDICATOR] [TABLE] SI. | No. | Particulars | PARTI | STANDALONE | Three Months | Ended | Previous Three | Months Ended | Corresp. Three | Mont...
Document 2 (Income Statement): [INCOME_INDICATOR] [TABLE] Sr. | Particulars | Three Months Ended | Year Ended | Mar 31, 2018 | Dec 31,2017 | Mar 31, 2017 | Dec 31,2017 | Unaudited |...
Document 3 (Income Statement): [INCOME_INDICATOR] [TABLE] Particulars | Standalone | Consolidated | Quarter Fnded | Year Ended | V<lir l-ixl«l | 31.03.2013 | 31.12.201.' | 31.03.201...

Data loaded successfully


In [7]:
def augment_cash_flow_data(documents, labels, target_samples=100):
    """
    Data augmentation specifically for Cash Flow minority class
    Cash Flow low recall by increasing training samples from 36 to ~100
    """
    augmented_docs = documents.copy()
    augmented_labels = labels.copy()

    # Find Cash Flow samples (label = 2)
    cash_flow_indices = [i for i, label in enumerate(labels) if label == 2]
    cash_flow_docs = [documents[i] for i in cash_flow_indices]

    print(f"Found {len(cash_flow_docs)} original Cash Flow samples")
    print(f"Augmenting to reach {target_samples} samples...")

    if len(cash_flow_docs) == 0:
        print("No Cash Flow samples found to augment!")
        return documents, labels

    # Simple augmentation techniques for text
    augmentation_techniques = [
        lambda text: text.replace('[NUMBER]', '[AMOUNT]'),  # Synonym replacement
        lambda text: text.replace('[CURRENCY]', '[DOLLAR_AMOUNT]'),
        lambda text: text.replace('[DATE]', '[PERIOD]'),
        lambda text: re.sub(r'\boperating activities\b', 'operational activities', text, flags=re.I),
        lambda text: re.sub(r'\binvesting activities\b', 'investment activities', text, flags=re.I),
        lambda text: re.sub(r'\bfinancing activities\b', 'financial activities', text, flags=re.I),
        lambda text: text.replace('cash provided by', 'cash generated from'),
        lambda text: text.replace('cash used in', 'cash applied to'),
    ]

    samples_needed = target_samples - len(cash_flow_docs)

    for i in range(samples_needed):
        # Select random original Cash Flow document
        base_doc = random.choice(cash_flow_docs)

        # Apply random augmentation
        technique = random.choice(augmentation_techniques)
        augmented_doc = technique(base_doc)

        # Add some variation by combining techniques
        if random.random() > 0.7:  # 30% chance of double augmentation
            technique2 = random.choice(augmentation_techniques)
            augmented_doc = technique2(augmented_doc)

        # Ensure augmented doc is different from original
        if augmented_doc != base_doc and augmented_doc not in augmented_docs:
            augmented_docs.append(augmented_doc)
            augmented_labels.append(2)  # Cash Flow label

    print(f"Added {len(augmented_docs) - len(documents)} augmented Cash Flow samples")

    return augmented_docs, augmented_labels

print("Applying Cash Flow data augmentation")
documents, labels = augment_cash_flow_data(documents, labels, target_samples=100)

label_dist = Counter(labels)
print("\nClass Distribution After Augmentation:")
for label_name, label_id in label_map.items():
    print(f"  {label_name}: {label_dist[label_id]}")

print("\nData augmentation applied")

Applying Cash Flow data augmentation
Found 36 original Cash Flow samples
Augmenting to reach 100 samples...
Added 24 augmented Cash Flow samples

Class Distribution After Augmentation:
  Income Statement: 317
  Balance Sheets: 282
  Cash Flow: 60
  Notes: 702
  Others: 1236

Data augmentation applied


In [8]:
class FocalLoss(torch.nn.Module):
    """
    Focal Loss for handling extreme class imbalance
    """
    def __init__(self, alpha=1, gamma=2, weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.weight = weight

    def forward(self, inputs, targets):
        # Standard cross entropy
        ce_loss = torch.nn.functional.cross_entropy(inputs, targets, reduction='none', weight=self.weight)

        # Focal loss modification: focus on hard examples
        pt = torch.exp(-ce_loss)  # probability of correct class
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()

class FinancialDocumentDataset(Dataset):
    """
    PyTorch Dataset for financial documents
    """
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenize
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def create_stratified_splits(documents, labels, test_size=0.2, val_size=0.1, random_state=42):
    """
    Create stratified train/val/test splits to maintain class distribution
    """
    # First split: train+val vs test
    X_temp, X_test, y_temp, y_test = train_test_split(
        documents, labels,
        test_size=test_size,
        random_state=random_state,
        stratify=labels
    )

    # Second split: train vs val
    val_ratio = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_ratio,
        random_state=random_state,
        stratify=y_temp
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

def initialize_bert_model(num_labels=5, model_name='bert-base-uncased'):
    """
    Initialize BERT model for sequence classification
    """
    model = BertForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        output_attentions=False,
        output_hidden_states=False
    )

    tokenizer = BertTokenizer.from_pretrained(model_name)

    return model, tokenizer

print("Model classes and functions defined")

Model classes and functions defined


In [9]:
# Calculate class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class Weights After Augmentation:")
label_names = list(label_map.keys())
for label_id, weight in enumerate(class_weights):
    print(f"  {label_names[label_id]}: {weight:.4f}")

# Create stratified splits
print("\nCreating stratified train/val/test splits...")
X_train, X_val, X_test, y_train, y_val, y_test = create_stratified_splits(
    documents, labels, test_size=0.2, val_size=0.1
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

# Initialize model and tokenizer
print("\nInitializing BERT model...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model, tokenizer = initialize_bert_model(num_labels=len(label_map))

print("\nData prepared for training")


Class Weights After Augmentation:
  Income Statement: 1.6385
  Balance Sheets: 1.8418
  Cash Flow: 8.6567
  Notes: 0.7399
  Others: 0.4202

Creating stratified train/val/test splits...
Training samples: 1817
Validation samples: 260
Test samples: 520

Initializing BERT model...
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


Data prepared for training


In [10]:
def evaluate_model_with_class_metrics(model, data_loader, device):
    """
    Evaluate model with per-class metrics for minority class monitoring
    """
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    accuracy = np.sum(np.array(predictions) == np.array(true_labels)) / len(true_labels)

    # Calculate per-class recall scores
    class_recalls = recall_score(true_labels, predictions, labels=[0,1,2,3,4], average=None, zero_division=0)

    return accuracy, avg_loss, class_recalls

def train_bert_classifier(model, train_loader, val_loader, device, epochs=4, learning_rate=2e-5, class_weights=None):
    """
    IMPROVED: Train BERT classifier with Focal Loss and minority class monitoring
    """
    # Use Focal Loss instead of standard CrossEntropyLoss
    if class_weights is not None:
        loss_fn = FocalLoss(alpha=1, gamma=2, weight=class_weights.to(device))
        print("Using Focal Loss with class weights for better minority class handling")
    else:
        loss_fn = FocalLoss(alpha=1, gamma=2)
        print("Using Focal Loss without class weights")

    # Optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate,
        eps=1e-8,
        weight_decay=0.01
    )

    # Learning rate scheduler
    from transformers import get_linear_schedule_with_warmup
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    model.to(device)
    best_val_accuracy = 0
    best_cash_flow_recall = 0

    for epoch in range(epochs):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch + 1}/{epochs}")
        print(f"{'='*50}")

        # Training phase
        model.train()
        total_train_loss = 0

        for batch in tqdm(train_loader, desc="Training"):
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss = loss_fn(logits, labels)
            total_train_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        avg_train_loss = total_train_loss / len(train_loader)
        print(f"\nAverage Training Loss: {avg_train_loss:.4f}")

        # Validation with detailed metrics
        val_accuracy, val_loss, class_recalls = evaluate_model_with_class_metrics(model, val_loader, device)
        print(f"Validation Loss: {val_loss:.4f}")
        print(f"Validation Accuracy: {val_accuracy:.4f}")

        # Display per-class recalls
        class_names = ['Income Statement', 'Balance Sheets', 'Cash Flow', 'Notes', 'Others']
        for i, recall in enumerate(class_recalls):
            print(f"  {class_names[i]} Recall: {recall:.4f}")

        # Save model based on Cash Flow performance too
        cash_flow_recall = class_recalls[2] if len(class_recalls) > 2 else 0
        if val_accuracy > best_val_accuracy or cash_flow_recall > best_cash_flow_recall:
            if cash_flow_recall > best_cash_flow_recall:
                best_cash_flow_recall = cash_flow_recall
                print(f"✓ New best Cash Flow recall: {cash_flow_recall:.4f}")
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy

            # Save complete model state
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'val_accuracy': val_accuracy,
                'cash_flow_recall': cash_flow_recall,
                'class_recalls': class_recalls
            }, 'best_bert_financial_classifier_improved.pt')
            print("✓ Best model saved!")

    return model

print("Training functions defined")

Training functions defined


In [11]:
MAX_LENGTH = 256
BATCH_SIZE = 4

print("Creating PyTorch datasets..")
train_dataset = FinancialDocumentDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset = FinancialDocumentDataset(X_val, y_val, tokenizer, MAX_LENGTH)
test_dataset = FinancialDocumentDataset(X_test, y_test, tokenizer, MAX_LENGTH)

print("Creating dataloaders..")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

print("\nDatasets and dataloaders ready")

Creating PyTorch datasets..
Creating dataloaders..
Training batches: 455
Validation batches: 65
Test batches: 130

Datasets and dataloaders ready


In [12]:
print("- Focal Loss for class imbalance")
print("- Cash Flow data augmentation")
print("- Improved HTML preprocessing")
print("- Per-class performance monitoring")
print("="*60)

EPOCHS = 4
LEARNING_RATE = 2e-5

model = train_bert_classifier(
    model, train_loader, val_loader, device,
    epochs=EPOCHS, learning_rate=LEARNING_RATE, class_weights=class_weights
)

tokenizer.save_pretrained('financial_tokenizer_improved')

print("\nTRAINING COMPLETE.")
print("Model saved as 'best_bert_financial_classifier_improved.pt'")

- Focal Loss for class imbalance
- Cash Flow data augmentation
- Improved HTML preprocessing
- Per-class performance monitoring
Using Focal Loss with class weights for better minority class handling

Epoch 1/4


Training: 100%|██████████| 455/455 [01:35<00:00,  4.75it/s]



Average Training Loss: 0.6286
Validation Loss: 0.4223
Validation Accuracy: 0.9000
  Income Statement Recall: 1.0000
  Balance Sheets Recall: 0.9643
  Cash Flow Recall: 0.6667
  Notes Recall: 0.9714
  Others Recall: 0.8306
✓ New best Cash Flow recall: 0.6667
✓ Best model saved!

Epoch 2/4


Training: 100%|██████████| 455/455 [01:31<00:00,  4.97it/s]



Average Training Loss: 0.0979
Validation Loss: 0.4357
Validation Accuracy: 0.8731
  Income Statement Recall: 1.0000
  Balance Sheets Recall: 0.9643
  Cash Flow Recall: 0.8333
  Notes Recall: 0.9571
  Others Recall: 0.7742
✓ New best Cash Flow recall: 0.8333
✓ Best model saved!

Epoch 3/4


Training: 100%|██████████| 455/455 [01:34<00:00,  4.83it/s]



Average Training Loss: 0.0425
Validation Loss: 0.2235
Validation Accuracy: 0.9346
  Income Statement Recall: 1.0000
  Balance Sheets Recall: 0.9643
  Cash Flow Recall: 1.0000
  Notes Recall: 0.9714
  Others Recall: 0.8871
✓ New best Cash Flow recall: 1.0000
✓ Best model saved!

Epoch 4/4


Training: 100%|██████████| 455/455 [01:34<00:00,  4.82it/s]



Average Training Loss: 0.0209
Validation Loss: 0.2209
Validation Accuracy: 0.9500
  Income Statement Recall: 1.0000
  Balance Sheets Recall: 0.9643
  Cash Flow Recall: 1.0000
  Notes Recall: 0.9429
  Others Recall: 0.9355
✓ Best model saved!

TRAINING COMPLETE.
Model saved as 'best_bert_financial_classifier_improved.pt'


In [13]:
def detailed_evaluation(model, test_loader, device, label_names):
    """
    Generate detailed classification report and confusion matrix
    """
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())

    # Classification report
    print("\n" + "="*60)
    print("="*60)
    print(classification_report(
        true_labels,
        predictions,
        target_names=label_names,
        digits=4
    ))

    # Confusion matrix
    print("\n" + "="*60)
    print("CONFUSION MATRIX")
    print("="*60)
    cm = confusion_matrix(true_labels, predictions)
    print(cm)

    return predictions, true_labels

print("Loading best trained model...")
checkpoint = torch.load('best_bert_financial_classifier_improved.pt', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Model loaded successfully.")
print(f"Best validation accuracy: {checkpoint.get('val_accuracy', 'N/A'):.4f}")
print(f"Best Cash Flow recall: {checkpoint.get('cash_flow_recall', 'N/A'):.4f}")

# Final evaluation
predictions, true_labels = detailed_evaluation(model, test_loader, device, label_names)

print("\nFinal evaluation done")

Loading best trained model...
Model loaded successfully.
Best validation accuracy: 0.9500
Best Cash Flow recall: 1.0000

                  precision    recall  f1-score   support

Income Statement     0.9516    0.9365    0.9440        63
  Balance Sheets     0.9649    0.9821    0.9735        56
       Cash Flow     0.8571    1.0000    0.9231        12
           Notes     0.9441    0.9574    0.9507       141
          Others     0.9590    0.9435    0.9512       248

        accuracy                         0.9519       520
       macro avg     0.9353    0.9639    0.9485       520
    weighted avg     0.9523    0.9519    0.9519       520


CONFUSION MATRIX
[[ 59   0   0   0   4]
 [  0  55   0   0   1]
 [  0   0  12   0   0]
 [  0   0   1 135   5]
 [  3   2   1   8 234]]

Final evaluation done
